# Flight Price — Economy Class — Cleaning Log & Quick-Revision Reference

This notebook records every cleaning and feature-engineering step performed on the Economy class
flight dataset (206,774 rows, 11 columns originally — part of the Kaggle `shubhambathwal/flight-price-prediction`
dataset, split by class).
For each step: the issue found, the code used, why the fix was made, and how it was verified.

**Note:** No `days_left` column here, unlike the Business notebook — it couldn't be reliably attached
from `Clean_Dataset.csv` (the join was ~85% ambiguous, since the same flight/price combination recurs
across many different dates with no way to disambiguate). Excluded rather than guessed.

## **0. Load & Connect**

**Note:** Raw data is a plain CSV (`economy_raw.csv`).

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('economy_raw.csv')

In [3]:
df_copy = df.copy()

## **1. Initial Inspection**

**Check:** Shape, columns, dtypes, and null/duplicate counts before touching anything.

**Result:** 206,774 rows × 11 columns, all `object` dtype except numeric fields not yet parsed. Zero
nulls, but (unlike Business) **2 duplicate rows** found — addressed in Section 3.

In [4]:
df_copy.shape

(206774, 11)

In [5]:
df_copy.columns

Index(['date', 'airline', 'ch_code', 'num_code', 'dep_time', 'from',
       'time_taken', 'stop', 'arr_time', 'to', 'price'],
      dtype='str')

In [6]:
df_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 206774 entries, 0 to 206773
Data columns (total 11 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   date        206774 non-null  str  
 1   airline     206774 non-null  str  
 2   ch_code     206774 non-null  str  
 3   num_code    206774 non-null  int64
 4   dep_time    206774 non-null  str  
 5   from        206774 non-null  str  
 6   time_taken  206774 non-null  str  
 7   stop        206774 non-null  str  
 8   arr_time    206774 non-null  str  
 9   to          206774 non-null  str  
 10  price       206774 non-null  str  
dtypes: int64(1), str(10)
memory usage: 17.4 MB


## **2. `num_code` — Digit-Length Check**

**Check:** Business class had a consistent 3-digit `num_code` for every row — verified this does **not**
carry over to Economy before assuming it. Result: Economy has a mix of 3-digit and 4-digit codes,
confirming the two datasets shouldn't be assumed identical in structure just because they share a schema.

In [7]:
df_copy[df_copy['num_code'] > 999]

,date,airline,ch_code,num_code,dep_time,from,time_taken,stop,arr_time,to,price
0,11-02-2022,SpiceJet,SG,8709,18:55,Delhi,02h 10m,non-stop,21:05,Mumbai,"5,953"
1,11-02-2022,SpiceJet,SG,8157,06:20,Delhi,02h 20m,non-stop,08:40,Mumbai,"5,953"
12,11-02-2022,Indigo,6E,5001,07:15,Delhi,02h 10m,non-stop,09:25,Mumbai,"5,955"
13,11-02-2022,Indigo,6E,6202,12:00,Delhi,02h 10m,non-stop,14:10,Mumbai,"5,955"
15,11-02-2022,Indigo,6E,6278,08:45,Delhi,02h 20m,non-stop,11:05,Mumbai,"5,955"
...,...,...,...,...,...,...,...,...,...,...,...
206719,31-03-2022,Indigo,6E,6006,15:50,Chennai,01h 15m,non-stop,17:05,Hyderabad,"1,551"
206722,31-03-2022,Indigo,6E,6215,18:10,Chennai,01h 20m,non-stop,19:30,Hyderabad,"1,551"
206724,31-03-2022,AirAsia,I5,1229,23:30,Chennai,09h 15m,1-stop\n\t\t\t\t\t\t\t\t\t\t\t\t\n\t\t\t\t\t\t...,08:45,Hyderabad,"1,550"
206725,31-03-2022,AirAsia,I5,2462,22:35,Chennai,10h 10m,1-stop\n\t\t\t\t\t\t\t\t\t\t\t\t\n\t\t\t\t\t\t...,08:45,Hyderabad,"1,550"


In [8]:
df_copy.isnull().sum()

date          0
airline       0
ch_code       0
num_code      0
dep_time      0
from          0
time_taken    0
stop          0
arr_time      0
to            0
price         0
dtype: int64

## **3. Duplicate Rows**

**Issue:** 2 exact duplicate rows found (Business had zero) — dropped.

In [9]:
dup_rows = df_copy.duplicated().sum()
print(f"duplicated rows: {dup_rows}")

df_copy = df_copy.drop_duplicates()

if df_copy.duplicated().sum() == 0:
    print(f"Successfully done!, duplicates left: {df_copy.duplicated().sum()}")

duplicated rows: 2


Successfully done!, duplicates left: 0


## **4. Zero-Variance & ID Columns (nunique check)**

**Check:** Ran `nunique()` across all columns.

**Result:**
- `airline`, `ch_code` — **8** unique values (vs Business's 2 — Economy is flown by many more carriers)
- `from`, `to` — 6 cities each, same as Business
- `num_code`, `dep_time`, `time_taken`, `stop`, `arr_time`, `price` — all need parsing (addressed below)

In [10]:
cols = df.columns

for col in cols:
    print(col, df[col].nunique())

date 49
airline 8
ch_code 8
num_code 1254
dep_time 251
from 6
time_taken 483
stop 37
arr_time 266
to 6
price 9819


## **5. `num_code` → `flight_number`**

**Issue:** Same misleading name as Business — `num_code` is actually the numeric part of the flight number.

**Fix:** Renamed to `flight_number`.

In [11]:
df_copy = df_copy.rename(columns={'num_code': 'flight_number'})

## **6. `date` — String to Datetime**

**Issue:** `date` was a plain string in `DD-MM-YYYY` format.

**Fix:** Converted with `pd.to_datetime(format='%d-%m-%Y')`, verified the resulting dtype.

In [12]:
df_copy['date'] = pd.to_datetime(df['date'], format= '%d-%m-%Y')
print(df_copy['date'].dtype)

if df_copy['date'].dtype == 'datetime64[ns]':
    print('Successfully done')
else:
    print('failed!')

datetime64[us]
failed!


## **7. `price` — String to Numeric**

**Issue:** `price` was a string with thousands-separator commas.

**Fix:** Stripped the comma, cast to `int`.

In [13]:
df_copy['price'] = df['price'].astype(str).str.replace(',','').astype(int)

print(df_copy['price'].sample(5))

116909     4672
23662      7535
62881      9879
121013    10059
108268     3545
Name: price, dtype: int64


## **8. Whitespace Check**

**Check:** Swept every text column for whitespace issues.

**Result:** Only `stop` affected — 194,567 rows (94%) had embedded `\n\t\t\t` garbage, same pattern as
Business but at a larger scale.

**Fix:** Same regex approach — collapse all whitespace runs, not just leading/trailing.

In [14]:
text_cols = df_copy.select_dtypes(include='object').columns

print("WhiteSpaces in: ")

dirty_col = []

for col in text_cols:
    dirty = (df_copy[col].astype(str) != df_copy[col].astype(str).str.strip()).sum()
    print(f'{col}: {dirty}')
    if dirty > 0:
        dirty_col.append(col)

print(f'\nColumns with whitespaces: {dirty_col}')

for col in dirty_col:
    df_copy[col] = df_copy[col].str.replace(r'\s+', ' ', regex=True).str.strip()

print('\nAfter cleaning:')
print('Whitespaces in:')
for col in dirty_col:
    dirty = (df_copy[col].astype(str) != df_copy[col].astype(str).str.strip()).sum()
    if dirty > 0:
        print(f'{col}: {dirty}')
print('successfully done if nothing printed above')

WhiteSpaces in: 


airline: 0
ch_code: 0


dep_time: 0
from: 0


/tmp/ipykernel_482/1679972277.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df_copy.select_dtypes(include='object').columns


time_taken: 0


stop: 194567


arr_time: 0
to: 0

Columns with whitespaces: ['stop']



After cleaning:
Whitespaces in:


successfully done if nothing printed above


## **9. `dep_time`, `arr_time`, `time_taken` → Numeric Features**

**Issue:** Same three raw-string columns as Business, same problem.

**Fix:**
- `dep_time` → `dep_hour` (int)
- `arr_time` → `arr_hour` (int)
- `time_taken` → `duration_mins` (int, regex-parsed)

Raw columns dropped once numeric versions exist.

In [15]:
import re

def parse_duration(s):
    h = re.search(r'(\d+)h', s)
    m = re.search(r'(\d+)m', s)
    hours = int(h.group(1)) if h else 0
    mins = int(m.group(1)) if m else 0
    return hours * 60 + mins

df_copy['dep_hour'] = df_copy['dep_time'].str.split(':').str[0].astype(int)
df_copy['arr_hour'] = df_copy['arr_time'].str.split(':').str[0].astype(int)
df_copy['duration_mins'] = df_copy['time_taken'].apply(parse_duration)

df_copy.drop(columns=['dep_time', 'arr_time', 'time_taken'], inplace=True)

df_copy.sample(5)

,date,airline,ch_code,flight_number,from,stop,to,price,dep_hour,arr_hour,duration_mins
129413,2022-02-24,Vistara,UK,720,Kolkata,1-stop,Mumbai,11139,7,21,845
123150,2022-03-06,AirAsia,I5,2473,Kolkata,1-stop,Delhi,3014,9,22,755
154259,2022-02-27,Vistara,UK,876,Hyderabad,1-stop,Delhi,6292,21,11,805
198729,2022-02-21,Indigo,6E,6567,Chennai,1-stop,Kolkata,9153,14,23,550
118701,2022-03-20,Vistara,UK,852,Bangalore,1-stop,Chennai,4672,9,8,1400


In [16]:
df_copy

,date,airline,ch_code,flight_number,from,stop,to,price,dep_hour,arr_hour,duration_mins
0,2022-02-11,SpiceJet,SG,8709,Delhi,non-stop,Mumbai,5953,18,21,130
1,2022-02-11,SpiceJet,SG,8157,Delhi,non-stop,Mumbai,5953,6,8,140
2,2022-02-11,AirAsia,I5,764,Delhi,non-stop,Mumbai,5956,4,6,130
3,2022-02-11,Vistara,UK,995,Delhi,non-stop,Mumbai,5955,10,12,135
4,2022-02-11,Vistara,UK,963,Delhi,non-stop,Mumbai,5955,8,11,140
...,...,...,...,...,...,...,...,...,...,...,...
206769,2022-03-31,Vistara,UK,832,Chennai,1-stop,Hyderabad,7697,7,20,830
206770,2022-03-31,Vistara,UK,832,Chennai,1-stop,Hyderabad,7709,7,20,830
206771,2022-03-31,Vistara,UK,826,Chennai,1-stop,Hyderabad,8640,12,9,1235
206772,2022-03-31,Vistara,UK,822,Chennai,1-stop,Hyderabad,8640,9,9,1400


## **10. `duration_mins` — Outlier Sanity Check**

**Check:** Reviewed `duration_mins > 2000` rows, same as Business — confirmed genuine long-layover
connections, not parsing errors. Sample size thins out fast past ~2000 mins (a handful of flights per
exact-minute value) — worth bucketing rather than trusting exact-minute medians at that tail during EDA.

In [17]:
df_copy[df_copy['duration_mins'] > 2000].head(10)

,date,airline,ch_code,flight_number,from,stop,to,price,dep_hour,arr_hour,duration_mins
10542,2022-02-14,Air India,AI,9887,Delhi,2+-stop,Bangalore,12321,5,18,2215
20466,2022-02-13,Vistara,UK,815,Delhi,2+-stop,Kolkata,17462,8,19,2150
20469,2022-02-13,Vistara,UK,801,Delhi,2+-stop,Kolkata,18927,9,19,2090
29419,2022-02-17,Air India,AI,481,Delhi,2+-stop,Hyderabad,10474,8,21,2215
29420,2022-02-17,Air India,AI,435,Delhi,2+-stop,Hyderabad,10474,5,21,2380
29536,2022-02-18,Air India,AI,481,Delhi,2+-stop,Hyderabad,10474,8,21,2215
29911,2022-02-21,Air India,AI,435,Delhi,2+-stop,Hyderabad,10474,5,21,2380
30302,2022-02-24,Air India,AI,481,Delhi,2+-stop,Hyderabad,9831,8,21,2215
30303,2022-02-24,Air India,AI,435,Delhi,2+-stop,Hyderabad,9831,5,21,2380
35404,2022-02-11,Vistara,UK,801,Delhi,2+-stop,Chennai,18757,9,19,2090


## **11. `stop` Column — Extract Count & Via-City**

**Issue:** Same three-in-one messy column as Business.

**Fix:** Same approach — `stop_count` via prefix check, `via_city` via regex, raw column dropped.

**Note:** Coverage is much sparser here than Business — only ~2.5% of stopped Economy flights actually
have a via-city captured (vs the source simply not recording it for the rest). Decide fill strategy
(`Direct` vs `Not_Recorded`) at modeling time, not here.

In [18]:
import re

bef_shape = df_copy.shape[1]
print(f'total_columns before any operation: {bef_shape}')

def stop_count(s):
    s = s.strip()
    if s.startswith('non-stop'):
        return 0
    elif s.startswith('2+'):
        return 2
    else:
        return 1

df_copy['stop_count'] = df_copy['stop'].apply(stop_count)

print(f"Successfully added!, total columns after adding stop_count: {df_copy.shape[1]}")

def extract_via_city(s):
    match = re.search(r'Via\s+([A-Za-z\s]+)', s)
    return match.group(1).strip() if match else None

df_copy['via_city'] = df_copy['stop'].apply(extract_via_city)

print(f"Successfully added!, total columns after adding via_city: {df_copy.shape[1]}")

# dropping stop column
df_copy.drop(columns=['stop'],inplace=True)

print(f"Successfully removed!, total columns affter removing stop: {df_copy.shape[1]}")

total_columns before any operation: 11
Successfully added!, total columns after adding stop_count: 12
Successfully added!, total columns after adding via_city: 13
Successfully removed!, total columns affter removing stop: 12


## **12. `from` & `to` — Combined into `route`**

**Fix:** Built `route` = `from` + '-' + `to`, same as Business.

In [19]:
df_copy['route'] = df['from'] + '-' + df['to']

## **13. `dep_hour` / `arr_hour` — Binned into Day-Cycle Periods**

**Fix:** Same as Business — `dep_period`/`arr_period` added alongside raw hour columns, both kept for
different model types (bucketed for Linear Regression, raw for tree-based models).

In [20]:
def time_bucket(hour):
    if 0 <= hour < 6:
        return 'Late_Night'
    elif 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 20:
        return 'Evening'
    else:
        return 'Night'

df_copy['dep_period'] = df_copy['dep_hour'].apply(time_bucket)
df_copy['arr_period'] = df_copy['arr_hour'].apply(time_bucket)

## **14. Final Column Order & Export**

**Note:** No `days_left` in the order list, unlike Business — genuinely absent, not an oversight.

In [21]:
order = ['date', 'airline', 'ch_code', 'flight_number', 'from', 'to', 'via_city', 'route',
       'stop_count', 'arr_hour','arr_period', 'dep_hour', 'dep_period', 'duration_mins',
       'price']

df_copy = df_copy[order]
df_copy.sample(10)

,date,airline,ch_code,flight_number,from,to,via_city,route,stop_count,arr_hour,arr_period,dep_hour,dep_period,duration_mins,price
5416,2022-03-10,Indigo,6E,152,Delhi,Mumbai,NaN,Delhi-Mumbai,1,17,Evening,10,Morning,460,3968
139833,2022-03-13,Vistara,UK,738,Kolkata,Bangalore,NaN,Kolkata-Bangalore,1,8,Morning,18,Evening,835,8111
154276,2022-02-27,GO FIRST,G8,123,Hyderabad,Delhi,NaN,Hyderabad-Delhi,1,16,Afternoon,6,Morning,650,6132
106854,2022-03-12,Vistara,UK,818,Bangalore,Kolkata,NaN,Bangalore-Kolkata,1,9,Morning,19,Evening,835,8112
76644,2022-03-27,AirAsia,I5,767,Mumbai,Hyderabad,NaN,Mumbai-Hyderabad,2,20,Night,7,Morning,785,2105
14214,2022-03-04,Air India,AI,883,Delhi,Bangalore,NaN,Delhi-Bangalore,1,22,Night,22,Night,1470,8423
5378,2022-03-10,Indigo,6E,5041,Delhi,Mumbai,NaN,Delhi-Mumbai,0,20,Night,18,Evening,130,3001
9490,2022-03-29,Vistara,UK,833,Delhi,Mumbai,NaN,Delhi-Mumbai,1,14,Afternoon,7,Morning,440,4691
181838,2022-02-22,Vistara,UK,824,Chennai,Delhi,NaN,Chennai-Delhi,1,18,Evening,20,Night,1295,11448
109035,2022-03-27,Vistara,UK,858,Bangalore,Kolkata,NaN,Bangalore-Kolkata,1,16,Afternoon,6,Morning,615,6271


In [22]:
df_copy.to_csv('economy_cleaned.csv', index=False)